In [ ]:
import hashlib
import json
import time
from datetime import datetime

In [ ]:
class Block:
    def __init__(self, index, previous_hash, timestamp, data, nonce=None, difficulty=None):
        self.index = index
        self.previous_hash = previous_hash
        self.timestamp = timestamp
        self.data = data
        self.nonce = nonce
        self.difficulty = difficulty
        self.hash = None

    def compute_hash(self):
        block_dict = dict(self.__dict__)
        if 'hash' in block_dict:
            del block_dict['hash']
        block_string = json.dumps(block_dict, sort_keys=True).encode()
        return hashlib.sha256(block_string).hexdigest()

In [ ]:
class Blockchain:
    def __init__(self, difficulty=4, target_block_time=2):
        self.difficulty = difficulty
        self.target_block_time = target_block_time
        self.unconfirmed_transactions = []
        self.chain = []
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis_block = Block(0, None, str(datetime.now()), "Genesis Block", 0, self.difficulty)
        genesis_block.hash = genesis_block.compute_hash()
        self.chain.append(genesis_block)

    def get_last_block(self):
        return self.chain[-1]

    def add_block(self, block, new_hash):
        previous_hash = self.get_last_block().hash
        if previous_hash != block.previous_hash:
            return False
        if not self.is_valid_proof(block, new_hash):
            return False
        block.hash = new_hash
        self.chain.append(block)
        return True

    def is_valid_proof(self, block, block_hash):
        return (block_hash.startswith('0' * block.difficulty) and
                block_hash == block.compute_hash())

    def proof_of_work(self, block):
        block.nonce = 0
        computed_hash = block.compute_hash()
        while not computed_hash.startswith('0' * block.difficulty):
            block.nonce += 1
            computed_hash = block.compute_hash()
        return computed_hash

    def add_new_transaction(self, transaction):
        self.unconfirmed_transactions.append(transaction)

    def adjust_difficulty(self, mining_time):
        if mining_time < self.target_block_time / 2:
            self.difficulty += 1
        elif mining_time > self.target_block_time * 2:
            self.difficulty = max(1, self.difficulty - 1)

    def mine(self):
        if not self.unconfirmed_transactions:
            return False

        last_block = self.get_last_block()
        new_block = Block(index=last_block.index + 1,
                          previous_hash=last_block.hash,
                          timestamp=str(datetime.now()),
                          data=self.unconfirmed_transactions,
                          difficulty=self.difficulty)

        start_time = time.time()
        proof = self.proof_of_work(new_block)
        mining_time = time.time() - start_time

        if not self.add_block(new_block, proof):
            return False
        self.unconfirmed_transactions = []
        self.adjust_difficulty(mining_time)
        return new_block.index

    def is_chain_valid(self):
        result = True
        previous_hash = None

        for i, block in enumerate(self.chain):
            if i == 0:
                previous_hash = block.hash
                continue 

            block_hash = block.hash

            if not self.is_valid_proof(block, block_hash) or previous_hash != block.previous_hash:
                result = False
                break

            previous_hash = block_hash

        return result